# Family D — Router Training on Kaggle **RTX Pro 6000** (Blackwell, BF16) — v2

Confound-free Family-D run (supersedes the 2×T4 4-bit probe, kept as fallback).
96 GB single card → 30B in **BF16, no quantization**. `MODE` switch: `"probe"`
(100 steps, pick LR) → `"full"` (1 epoch = Family D).

**Blackwell setup ported from your `nemotron-train-4-0-all` cell 1** (offline):
- deps = the `ryanholbrook/nvidia-utility-script` **utility-script input**, mounted
  at `/kaggle/usr/lib/notebooks/...` (prebuilt torch/flash-attn/mamba/causal-conv1d
  + ptxas-blackwell); just `sys.path.insert` it — works offline.
- shadow `mamba_ssm.modules.mamba3` (its cutlass is sm_90 → crashes on sm_120).
- patch triton ptxas for arch ≥ 100.
- **force slow path** (`is_fast_path_available=False`): fast path's `causal_conv1d`
  is sm_90 → crashes on Blackwell. (This one is essential.)

Recipe parity with your originals: `MAX_SEQ=4096`, batch 1 × accum 8, LR 1e-4
cosine, 1 epoch, **full-sequence loss** (`MASK_PROMPT=False`, matching your
`DataCollatorForLanguageModeling`). Only difference vs A/B/C: we train the 23
`gate.weight` and nothing else.

In [ ]:
# 1. Allocator config (MUST be set before torch initializes CUDA) + offline deps.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
DEPS_DIR = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"  # utility-script input
assert os.path.isdir(DEPS_DIR), f"Add the utility script as input — not found: {DEPS_DIR}"
sys.path.insert(0, DEPS_DIR)
import torch
print("torch", torch.__version__, "| device:", torch.cuda.get_device_name(0),
      "| capability:", torch.cuda.get_device_capability(0),
      "| alloc_conf:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

In [ ]:
# 2. Blackwell init (ported from nemotron-train-4-0-all). Run BEFORE loading the model.
import os, sys, types, importlib, importlib.util, importlib.machinery, shutil

# (a) shadow mamba3 (cutlass sm_90 crashes on sm_120); keep real Triton ops.
for n in list(sys.modules):
    if n == "mamba_ssm" or n.startswith("mamba_ssm."): del sys.modules[n]
spec = importlib.util.find_spec("mamba_ssm")
pkg = types.ModuleType("mamba_ssm"); pkg.__path__ = [list(spec.submodule_search_locations)[0]]
pkg.__package__ = "mamba_ssm"
pkg.__spec__ = importlib.machinery.ModuleSpec("mamba_ssm", loader=None, is_package=True)
pkg.__spec__.submodule_search_locations = pkg.__path__
sys.modules["mamba_ssm"] = pkg
m3 = types.ModuleType("mamba_ssm.modules.mamba3"); m3.__package__ = "mamba_ssm.modules"
class FakeMamba3: pass
m3.Mamba3 = FakeMamba3; sys.modules["mamba_ssm.modules.mamba3"] = m3
print("mamba_ssm shadowed (mamba3 stubbed) ✓")

# (b) ptxas-blackwell from the util mount + patch triton for arch>=100.
src = f"{DEPS_DIR}/triton/backends/nvidia/bin/ptxas-blackwell"; dst = "/tmp/ptxas-blackwell"
shutil.copy2(src, dst); os.chmod(dst, 0o755)
import triton.backends.nvidia.compiler as _tnc
from triton.knobs import NvidiaTool
_orig = _tnc.get_ptxas
def _patched(arch):
    if arch >= 100:
        t = NvidiaTool.from_path(dst)
        if t: return t
    return _orig(arch)
_tnc.get_ptxas = _patched
print("ptxas-blackwell patched ✓")

In [ ]:
# 3. CONFIG
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1/"
DATA_PATH  = "/kaggle/input/datasets/corinakaiser/master-cot-train3/WONDERLAND_FINAL_MASTER.jsonl"
OUT_DIR    = "/kaggle/working/familyD"
MODE       = "probe"          # "probe" (100 steps, pick LR) -> "full" (1 epoch = Family D)
LR         = 1e-4             # probe: 1e-4 / 3e-5 / 1e-5 ; full: the chosen one
AUX_COEF   = 0.01
BALANCE_CAP= 0.08
MAX_LEN    = 2048            # probe at 2048; try 4096 for the full run (alloc fix may let it fit)
MASK_PROMPT= False           # False = full-sequence loss (matches your LoRA recipe)
PER_DEV_BATCH, GRAD_ACCUM, SEED = 1, 8, 42
MOE_LAYERS=[1,3,6,8,10,13,15,17,20,22,24,27,29,31,34,36,38,40,43,45,47,49,51]
import os; os.makedirs(OUT_DIR, exist_ok=True)
MAX_STEPS = 100 if MODE=="probe" else -1
EPOCHS    = 1

In [ ]:
# 4. Load model in BF16 on the single card; force slow path AFTER load.
import torch, torch.nn.functional as F
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          TrainingArguments, Trainer, TrainerCallback)
tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token
tok.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, low_cpu_mem_usage=True)
# (c) force slow path now that modeling_nemotron_h is imported (causal_conv1d sm_90 -> crash)
forced = 0
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        mod.is_fast_path_available = False; forced += 1
print(f"slow path forced on {forced} module(s) ✓")
model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()   # base frozen -> grad must reach the gates
print("loaded bf16 on", next(model.parameters()).device)

In [ ]:
# 5. Freeze all, unfreeze ONLY the 23 gate.weight (fp32 for router stability).
for p in model.parameters(): p.requires_grad_(False)
n_train = 0
for L in MOE_LAYERS:
    w = model.backbone.layers[L].mixer.gate.weight
    w.data = w.data.float(); w.requires_grad_(True); n_train += w.numel()
GATES = {L: model.backbone.layers[L].mixer.gate for L in MOE_LAYERS}
assert n_train > 0 and sum(p.requires_grad for p in model.parameters()) == len(MOE_LAYERS)
print(f"trainable router params: {n_train:,} across {len(GATES)} gates (rest frozen)")

In [ ]:
# 6. Load-balance aux loss (Switch form, FIXED for sigmoid router):
#    P_e from SOFTMAX over gate logits (proper distribution, sum=1) so aux tracks
#    *allocation* not gate sharpness. aux ~= 1 balanced, rises with concentration.
class BalanceHooks:
    def __init__(self, gates):
        self.store, self.handles = {}, []
        for L, g in gates.items(): self.handles.append(g.register_forward_hook(self._mk(L)))
    def _mk(self, L):
        def hook(m, inp, out): self.store[L] = inp[0].detach().reshape(-1, inp[0].shape[-1])
        return hook
    def clear(self): self.store = {}
    def compute(self, gates, dev):
        terms, max_load = [], 0.0
        for L, h in self.store.items():
            g = gates[L]; h = h.to(g.weight.device)
            logits = F.linear(h.float(), g.weight.float())        # [T, N] raw gate logits
            N = g.n_routed_experts
            P_e = logits.softmax(-1).mean(0)                       # [N] normalized prob (sum=1), differentiable
            with torch.no_grad():
                idx = logits.sigmoid().topk(g.top_k, dim=-1).indices   # base bias is 0; avoids meta-device buffer
                sel = torch.zeros(logits.shape[0], N, device=logits.device); sel.scatter_(1, idx, 1.0)
                tok_frac = sel.mean(0)                             # fraction of TOKENS selecting e (base max ~0.55)
                f_e = tok_frac / g.top_k                           # normalized load (sum=1)
                max_load = max(max_load, float(tok_frac.max()))    # collapse signal (unchanged meaning)
            terms.append((N * torch.sum(f_e * P_e)).to(dev))       # ~1 balanced, rises with concentration
        return (torch.stack(terms).mean() if terms else torch.zeros((), device=dev)), max_load
hooks = BalanceHooks(GATES)

In [ ]:
# 7. Dataset — render to text then tokenize (v5-safe: tokenize=True returns
#    tokenizers.Encoding objects, not int lists). Full-seq loss by default.
import json
from torch.utils.data import Dataset
class WonderlandSFT(Dataset):
    def __init__(self, path, tok, max_len, mask_prompt):
        self.rows = [json.loads(l)["messages"] for l in open(path) if l.strip()]
        self.tok, self.max_len, self.mask_prompt = tok, max_len, mask_prompt
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        m = self.rows[i]
        text = self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        full = self.tok(text, truncation=True, max_length=self.max_len,
                        add_special_tokens=False)["input_ids"]   # clean list[int]
        lab = list(full)
        if self.mask_prompt:
            ptext = self.tok.apply_chat_template(m[:-1], tokenize=False, add_generation_prompt=True)
            plen = len(self.tok(ptext, add_special_tokens=False)["input_ids"])
            for j in range(min(plen, len(lab))): lab[j] = -100
            if lab and all(x == -100 for x in lab): lab[-1] = full[-1]
        return {"input_ids": full, "labels": lab}
class PadCollator:
    def __init__(self, pad): self.pad = pad
    def __call__(self, batch):
        Lm = max(len(b["input_ids"]) for b in batch); ids, lab, att = [], [], []
        for b in batch:
            n = Lm - len(b["input_ids"])
            ids.append(list(b["input_ids"]) + [self.pad]*n)
            lab.append(list(b["labels"]) + [-100]*n)
            att.append([1]*len(b["input_ids"]) + [0]*n)
        return {"input_ids": torch.tensor(ids, dtype=torch.long),
                "labels": torch.tensor(lab, dtype=torch.long),
                "attention_mask": torch.tensor(att, dtype=torch.long)}
ds = WonderlandSFT(DATA_PATH, tok, MAX_LEN, MASK_PROMPT); print("examples:", len(ds))

In [ ]:
# 8. Train (BF16; aux loss + collapse monitor).
HIST = []
class RouterTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        hooks.clear(); out = model(**inputs); lm = out.loss
        aux, ml = hooks.compute(GATES, lm.device); loss = lm + AUX_COEF * aux
        self._last = (float(lm.detach()), float(aux.detach()), ml)
        return (loss, out) if return_outputs else loss
class BalanceMonitor(TrainerCallback):
    def __init__(self): self.bad = 0
    def on_log(self, args, state, control, **kw):
        if hasattr(trainer, "_last"):
            lm, aux, ml = trainer._last; HIST.append((state.global_step, lm, aux, ml))
            print(f"  step {state.global_step}: lm={lm:.4f} aux={aux:.4f} max_load={ml:.3f}", flush=True)
            self.bad = self.bad + 1 if ml > BALANCE_CAP else 0
            if self.bad >= 3:
                print(f"  !! COLLAPSE max_load>{BALANCE_CAP} — stopping"); control.should_training_stop = True
targs = TrainingArguments(output_dir=OUT_DIR, per_device_train_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, max_steps=MAX_STEPS,
    num_train_epochs=EPOCHS, lr_scheduler_type="cosine", warmup_ratio=0.05,
    optim="adamw_torch", weight_decay=0.0, max_grad_norm=1.0, bf16=True,
    logging_steps=5, save_strategy=("no" if MODE=="probe" else "steps"), save_steps=200,
    save_total_limit=3, seed=SEED, gradient_checkpointing=False, report_to="none",
    remove_unused_columns=False)
trainer = RouterTrainer(model=model, args=targs, train_dataset=ds,
                        data_collator=PadCollator(tok.pad_token_id))
trainer.add_callback(BalanceMonitor()); trainer.train()

In [ ]:
# 9. Save gate weights (= Family D when MODE='full') + plot.
import torch, json
gate_sd = {f"backbone.layers.{L}.mixer.gate.weight": GATES[L].weight.detach().cpu() for L in MOE_LAYERS}
torch.save(gate_sd, f"{OUT_DIR}/router_state_{MODE}_lr{LR}.pt")
json.dump(dict(mode=MODE, lr=LR, aux_coef=AUX_COEF, max_len=MAX_LEN, mask_prompt=MASK_PROMPT,
    trainable=sum(v.numel() for v in gate_sd.values())),
    open(f"{OUT_DIR}/familyD_config.json", "w"), indent=1)
import matplotlib.pyplot as plt
if HIST:
    s = [h[0] for h in HIST]; fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(s, [h[1] for h in HIST]); ax[0].set_title(f"LM loss (LR={LR}, {MODE})"); ax[0].set_xlabel("step")
    ax[1].plot(s, [h[3] for h in HIST]); ax[1].axhline(BALANCE_CAP, ls='--', c='r')
    ax[1].set_title("max expert load"); ax[1].set_xlabel("step"); plt.tight_layout(); plt.show()
print("saved", f"{OUT_DIR}/router_state_{MODE}_lr{LR}.pt")

## Run order
1. **Inputs:** add the base model, your Wonderland dataset, and
   `ryanholbrook/nvidia-utility-script` **as a utility script**. Internet OFF.
2. **Probe** (`MODE="probe"`): run LR 1e-4 / 3e-5 / 1e-5; pick the highest LR with
   falling loss + `max_load` under the cap.
3. **Full** (`MODE="full"`, chosen LR): 1 epoch → `router_state_full_*.pt` **is
   Family D**. Resume after a timeout via `trainer.train(resume_from_checkpoint=True)`.
4. Save it, tell me the LR + path; I load it `strict=False` and run the
   divergence + capability captures (§5.5).